# ACC Evolutionary Optimization — archived research results

This notebook is the **single public analysis entry point** for the Zenodo/GitHub release.
It reads the archived canonical RQ1–RQ4 evidence from `research_outputs/canonical/` and does not rerun the expensive optimization campaign.

Use `python main.py optimize` only if you actually want to generate a new local campaign.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = Path.cwd()
OUT = ROOT / "research_outputs" / "canonical"
if not OUT.exists():
    raise FileNotFoundError("research_outputs/canonical is missing")

status = json.loads((OUT / "evidence_status.json").read_text(encoding="utf-8"))
if status.get("issues"):
    raise RuntimeError(f"Archived evidence reports issues: {status['issues']}")

print("Canonical evidence loaded from:", OUT)
display(pd.DataFrame({
    "scope": ["RQ1", "RQ2", "RQ3", "RQ4"],
    "summary": [
        f"{status['RQ1']['controllers']} optimized controllers",
        f"{status['RQ2']['evaluations']} controller-profile evaluations",
        f"{status['RQ3']['evaluations']} evaluations across {status['RQ3']['conditions']} physical conditions",
        status['RQ4']['status'],
    ],
}))

## Study-level evidence summary

In [ ]:
display(Markdown((OUT / "RQ_evidence.md").read_text(encoding="utf-8")))

## RQ1 — optimizer quality, variability, convergence, and computational cost

The replication unit is the independent optimization run. Runtime and recorded function evaluations are reported as effort measures; this is not an equal-FE comparison.

In [ ]:
rq1 = OUT / "RQ1_optimizer"
for name in [
    "objective_summary.csv",
    "quality_cost_confidence.csv",
    "kruskal_wallis.csv",
    "independent_mann_whitney_corrected.csv",
    "representative_controllers.csv",
]:
    print("\n", name)
    display(pd.read_csv(rq1 / name))

for name in [
    "objective_distribution_boxplot.png",
    "convergence_median_iqr.png",
    "representative_cost_decomposition.png",
]:
    display(Image(filename=str(rq1 / name)))

## RQ2 — fixed-gain controller behavior

All optimized controllers are evaluated on synthetic diagnostics and profiles excluded from tuning. The archived tables include distribution summaries, transient behavior, recovery events, and exploratory training-to-held-out associations.

In [ ]:
rq2 = OUT / "RQ2_behavior"
for name in [
    "controller_distributions.csv",
    "directional_transient_distributions.csv",
    "recovery_events.csv",
    "training_heldout_spearman.csv",
]:
    print("\n", name)
    display(pd.read_csv(rq2 / name).head(40))

motion = rq2 / "representative_motion"
for name in [
    "cruise_control_periodic.png",
    "syn_follow_sinusoidal_motion.png",
    "v1_casale_motion.png",
    "v2_wltc_motion.png",
]:
    display(Image(filename=str(motion / name)))

## RQ3 — physical-condition sensitivity

Fixed controller gains are evaluated across the predeclared 12-condition grid. Comparisons are relative to each controller's own nominal result and use matched physical compensation.

In [ ]:
rq3 = OUT / "RQ3_physical"
for name in ["physical_distributions.csv", "median_degradation.csv"]:
    print("\n", name)
    display(pd.read_csv(rq3 / name).head(60))

for name in [
    "degradation_SYN_CRUISE_VSET_10X_cruise_speed_rmse_mps.png",
    "degradation_V1_CASALE_spacing_rmse_m.png",
    "degradation_V1_CASALE_positive_tractive_energy_wh_per_km.png",
]:
    display(Image(filename=str(rq3 / name)))

## RQ4 — objective-term ablation

The ablation uses paired canonical-DE runs for the full objective, no-jerk, no-energy, and tracking-only profiles. Compare common physical metrics and paired differences rather than unlike scalar fitness values.

In [ ]:
rq4 = OUT / "RQ4_objective"
for name in [
    "training_components_and_gains.csv",
    "paired_calibration_comparisons.csv",
    "ablation_distributions.csv",
    "paired_ablation_comparisons.csv",
]:
    print("\n", name)
    display(pd.read_csv(rq4 / name).head(60))

for name in [
    "ablation_SYN_CRUISE_VSET_10X_jerk_rms_mps3.png",
    "ablation_SYN_CRUISE_VSET_10X_positive_tractive_energy_wh_per_km.png",
    "ablation_V1_CASALE_spacing_rmse_m.png",
]:
    display(Image(filename=str(rq4 / name)))

## Interpretation boundaries

- The 1.5 s TTC line is a study-defined reporting threshold, not a universal safety guarantee.
- Positive tractive energy is a wheel-level demand surrogate, not fuel or battery consumption.
- RQ3 is limited to the declared 12 physical configurations with matched compensation.
- RQ4 conclusions are conditional on canonical DE and the frozen study protocol.
- Sensor noise/bias, explicit filtering, compensation mismatch, and equal-FE campaigns are outside the active study.